In [ ]:
"""比较不同城市层级中“买过3CE但未持续购买”人群的共同需求断点。

比较组：
1. 高线城市：一线城市 + 新一线城市；
2. 三四线城市：三线、四线及以下、三线及以下城市/县城。

重点检验：
- 个性化需求；
- 场景化需求；
- 对穿搭的兴趣；
- 对妆容的兴趣；
- 上述需求在同一受访者身上的重合程度；
- Q10B未持续购买原因与Q11B重新购买触发点。

使用：
    python 3CE_城市层级_未持续购买_共同需求断点分析.py 问卷数据.csv

依赖：pip install pandas openpyxl matplotlib
"""

In [ ]:
from __future__ import annotations

In [ ]:
import argparse
import re
from collections import Counter
from pathlib import Path

In [ ]:
import pandas as pd
from openpyxl.styles import Font, PatternFill

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

In [ ]:
GROUP = "城市比较组"
GROUP_ORDER = ["一线/新一线", "三四线/县城"]

In [ ]:
ALIASES: dict[str, list[str]] = {
    "id": ["记录ID", "答卷ID", "序号", "ID"],
    "age": ["年龄", "Q1. 您的年龄", "Q1 您的年龄"],
    "city": ["所在城市级别", "Q2. 您所在城市", "Q2 您所在城市"],
    "reason": ["购买彩妆的主要原因", "Q4. 您购买彩妆产品最主要的原因是什么", "您购买彩妆产品最主要的原因是什么"],
    "brand_need": ["最希望美妆品牌解决的问题", "Q8. 如果一个美妆品牌只能解决一个问题您最希望它帮助您", "您最希望它帮助您"],
    "q9": ["3CE了解程度", "Q9. 您对3CE的了解程度", "您对3CE的了解程度"],
    "q10b": ["Q10B～很少或未继续购买的原因", "Q10B. 您没有继续购买或很少购买3CE的主要原因是什么", "您没有继续购买或很少购买3CE的主要原因是什么"],
    "q11b": [
        "Q11B～重新关注或购买的方式",
        "Q11B. 以下哪些方式可能让您重新关注或购买3CE",
        "Q10B. 以下哪些方式可能让您重新关注或购买3CE",
        "以下哪些方式可能让您重新关注或购买3CE",
    ],
    "kdrama_element": ["影响兴趣的韩剧元素", "Q13. 韩剧中的哪些元素会影响您的兴趣", "韩剧中的哪些元素会影响您的兴趣"],
    "brand_info": ["希望品牌了解的信息", "Q15. 如果一个美妆品牌可以帮助您找到自己的风格您希望它了解哪些信息", "您希望它了解哪些信息"],
    "q16b": ["体验评分：场景妆容助手", "Q16B. 场景妆容助手", "场景妆容助手"],
    "q16c": ["体验评分：AI个人风格探索", "Q16C. AI个人风格探索", "AI个人风格探索"],
}

In [ ]:
OPTIONAL = {"id", "age"}

In [ ]:
# “个性化/场景化”使用多个题目的行为信号；修改这里即可调整主题口径。
PERSONALIZATION_PATTERNS: dict[str, str] = {
    "Q8品牌需要": r"适合自己|找到适合|个性化|个人风格",
    "Q10B流失原因": r"不清楚.*适合|色号.*适合|风格不再适合|其他品牌更懂|不适合自己",
    "Q11B重新激活": r"根据我的特征|推荐色号|虚拟试妆|AI妆容顾问|个人风格档案|个性化",
    "Q15希望品牌了解": r"性格|脸型|穿衣风格|兴趣爱好|影视角色|个人特征",
}
SCENARIO_PATTERNS: dict[str, str] = {
    "Q4购买原因": r"场合|场景|演唱会|面试|约会|旅行|聚会|通勤",
    "Q8品牌需要": r"生活场景|生活情景|不同场景|不同情景|不同生活.*形象|场景.*形象",
    "Q11B重新激活": r"生活场景|生活情景|场景.*妆容|情景.*妆容|妆容方案",
    "Q15希望品牌了解": r"生活场景|生活情景",
}

In [ ]:
def clean_text(value: object) -> str:
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

In [ ]:
def normalize_header(value: object) -> str:
    text = clean_text(value).lower().replace("？", "")
    # 问卷导出可能用“｜”“|”“～”“~”分隔题号和题目，匹配时统一忽略。
    return re.sub(r"[\s\-—_·•○□（）()，,。:：\"'“”‘’/|｜~～]+", "", text)

In [ ]:
def read_survey(path: Path, sheet_name: str | int = 0) -> pd.DataFrame:
    if path.suffix.lower() in {".xlsx", ".xlsm", ".xls"}:
        return pd.read_excel(path, sheet_name=sheet_name, dtype=str, keep_default_na=False)
    if path.suffix.lower() != ".csv":
        raise ValueError("只支持CSV或Excel文件。")
    errors: list[str] = []
    for encoding in ("utf-8-sig", "utf-8", "gb18030"):
        try:
            return pd.read_csv(path, encoding=encoding, dtype=str, keep_default_na=False)
        except (UnicodeDecodeError, pd.errors.ParserError) as exc:
            errors.append(f"{encoding}: {exc}")
    raise RuntimeError("无法读取CSV。\n" + "\n".join(errors))

In [ ]:
def resolve_columns(raw: pd.DataFrame) -> pd.DataFrame:
    actual = {normalize_header(column): column for column in raw.columns}
    rename: dict[object, str] = {}
    used: set[object] = set()
    missing: list[str] = []
    for key, aliases in ALIASES.items():
        source = next(
            (actual[normalize_header(alias)] for alias in aliases
             if normalize_header(alias) in actual and actual[normalize_header(alias)] not in used),
            None,
        )
        if source is None:
            candidates = []
            for normalized_actual, actual_column in actual.items():
                if actual_column in used:
                    continue
                if any(
                    len(normalize_header(alias)) >= 6
                    and (normalize_header(alias) in normalized_actual or normalized_actual in normalize_header(alias))
                    for alias in aliases
                ):
                    candidates.append(actual_column)
            if len(set(candidates)) == 1:
                source = candidates[0]
        if source is None:
            if key not in OPTIONAL:
                missing.append(aliases[0])
        else:
            used.add(source)
            rename[source] = key
    if missing:
        raise KeyError(
            "缺少分析所需字段：\n- " + "\n- ".join(missing)
            + "\n\n实际表头：\n- " + "\n- ".join(map(str, raw.columns))
        )
    df = raw.rename(columns=rename).copy()
    if "id" not in df:
        df["id"] = [f"ROW-{index + 2}" for index in range(len(df))]
    if "age" not in df:
        df["age"] = ""
    return df

In [ ]:
def city_group(value: object) -> str:
    s = re.sub(r"\s+", "", clean_text(value))
    if "新一线" in s or ("一线" in s and "新一线" not in s):
        return "一线/新一线"
    if re.search(r"三线及以下|三线以下|三四线|三线|四线|五线|六线|县城|低线城市", s):
        return "三四线/县城"
    return "不纳入"

In [ ]:
def is_lapsed_buyer(value: object) -> bool:
    """Q9中“买过1-2次”即购买后未持续留存分支。"""
    s = re.sub(r"\s+", "", clean_text(value))
    bought = bool(re.search(r"买过|购买过|1[-—–~～至到]2次", s))
    not_bought = bool(re.search(r"没买过|没有购买过|未购买过|从未购买|听说过.*未购买", s))
    explicitly_not_continued = bool(re.search(r"没有持续购买|未持续购买|未继续购买|不再购买", s))
    frequent = bool(re.search(r"经常购买|持续购买|一直购买|长期购买", s)) and not explicitly_not_continued
    return bought and not not_bought and not frequent

In [ ]:
def split_multi(value: object) -> list[str]:
    text = clean_text(value)
    if not text:
        return []
    parts = re.split(r"\s*(?:\||;|；|、|,|\n|\r|/|｜)\s*", text)
    return [part.strip(" []'\"") for part in parts if part.strip(" []'\"")]

In [ ]:
def contains_any(value: object, pattern: str) -> bool:
    return bool(re.search(pattern, clean_text(value), flags=re.IGNORECASE))

In [ ]:
def score_is_high(value: object) -> bool:
    match = re.search(r"([1-5])", clean_text(value))
    return bool(match and int(match.group(1)) >= 4)

In [ ]:
def code_themes(target: pd.DataFrame) -> pd.DataFrame:
    coded = target.copy()
    coded["个性化_Q8"] = coded["brand_need"].map(
        lambda value: contains_any(value, PERSONALIZATION_PATTERNS["Q8品牌需要"])
    )
    coded["个性化_Q10B"] = coded["q10b"].map(
        lambda value: contains_any(value, PERSONALIZATION_PATTERNS["Q10B流失原因"])
    )
    coded["个性化_Q11B"] = coded["q11b"].map(
        lambda value: contains_any(value, PERSONALIZATION_PATTERNS["Q11B重新激活"])
    )
    coded["个性化_Q15"] = coded["brand_info"].map(
        lambda value: contains_any(value, PERSONALIZATION_PATTERNS["Q15希望品牌了解"])
    )
    coded["个性化_Q16C高兴趣"] = coded["q16c"].map(score_is_high)

    coded["场景化_Q4"] = coded["reason"].map(
        lambda value: contains_any(value, SCENARIO_PATTERNS["Q4购买原因"])
    )
    coded["场景化_Q8"] = coded["brand_need"].map(
        lambda value: contains_any(value, SCENARIO_PATTERNS["Q8品牌需要"])
    )
    coded["场景化_Q11B"] = coded["q11b"].map(
        lambda value: contains_any(value, SCENARIO_PATTERNS["Q11B重新激活"])
    )
    coded["场景化_Q15"] = coded["brand_info"].map(
        lambda value: contains_any(value, SCENARIO_PATTERNS["Q15希望品牌了解"])
    )
    coded["场景化_Q16B高兴趣"] = coded["q16b"].map(score_is_high)

    coded["个性化需求"] = coded[[
        "个性化_Q8", "个性化_Q10B", "个性化_Q11B", "个性化_Q15", "个性化_Q16C高兴趣",
    ]].any(axis=1)
    coded["场景化需求"] = coded[[
        "场景化_Q4", "场景化_Q8", "场景化_Q11B", "场景化_Q15", "场景化_Q16B高兴趣",
    ]].any(axis=1)
    coded["穿搭兴趣"] = coded["kdrama_element"].map(lambda value: contains_any(value, r"穿搭"))
    coded["妆容兴趣"] = coded["kdrama_element"].map(lambda value: contains_any(value, r"妆容"))
    coded["个性化且场景化"] = coded["个性化需求"] & coded["场景化需求"]
    coded["穿搭或妆容兴趣"] = coded["穿搭兴趣"] | coded["妆容兴趣"]
    coded["穿搭且妆容兴趣"] = coded["穿搭兴趣"] & coded["妆容兴趣"]
    coded["核心重合画像"] = coded["个性化且场景化"] & coded["穿搭或妆容兴趣"]
    coded["四项严格重合"] = (
        coded["个性化需求"] & coded["场景化需求"] & coded["穿搭兴趣"] & coded["妆容兴趣"]
    )
    return coded

In [ ]:
METRICS = [
    "个性化需求", "场景化需求", "个性化且场景化", "穿搭兴趣", "妆容兴趣",
    "穿搭或妆容兴趣", "穿搭且妆容兴趣", "核心重合画像", "四项严格重合",
]

In [ ]:
def common_need_table(coded: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for metric in METRICS:
        group_values: dict[str, tuple[int, int, float]] = {}
        for group in GROUP_ORDER:
            subset = coded[coded[GROUP] == group]
            count = int(subset[metric].sum())
            base = len(subset)
            group_values[group] = (count, base, count / base if base else 0.0)
        high = group_values["一线/新一线"]
        low = group_values["三四线/县城"]
        rows.append([
            metric,
            high[0], high[2],
            low[0], low[2],
            high[2] >= 0.5 and low[2] >= 0.5,
        ])
    return pd.DataFrame(rows, columns=[
        "指标",
        "一线/新一线人数", "一线/新一线占比",
        "三四线/县城人数", "三四线/县城占比",
        "两组是否均过半",
    ])

In [ ]:
def source_signal_table(coded: pd.DataFrame) -> pd.DataFrame:
    source_columns = [
        "个性化_Q8", "个性化_Q10B", "个性化_Q11B", "个性化_Q15", "个性化_Q16C高兴趣",
        "场景化_Q4", "场景化_Q8", "场景化_Q11B", "场景化_Q15", "场景化_Q16B高兴趣",
    ]
    rows = []
    for source in source_columns:
        for group in GROUP_ORDER:
            subset = coded[coded[GROUP] == group]
            count = int(subset[source].sum())
            rows.append([source, group, count, len(subset), count / len(subset) if len(subset) else 0.0])
    return pd.DataFrame(rows, columns=["主题来源", GROUP, "命中人数", "组内分母", "组内命中率"])

In [ ]:
def grouped_multi_table(df: pd.DataFrame, field: str) -> pd.DataFrame:
    rows = []
    for group in GROUP_ORDER:
        subset = df[df[GROUP] == group]
        counter = Counter(option for value in subset[field] for option in split_multi(value))
        for option, count in counter.items():
            rows.append([group, option, count, len(subset), count / len(subset) if len(subset) else 0.0])
    out = pd.DataFrame(rows, columns=[GROUP, "选项", "选择人数", "组内分母", "组内选择率"])
    return out.sort_values([GROUP, "选择人数", "选项"], ascending=[True, False, True], ignore_index=True)

In [ ]:
def grouped_score_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for group in GROUP_ORDER:
        subset = df[df[GROUP] == group]
        for field, label in [("q16b", "Q16B 场景妆容助手"), ("q16c", "Q16C AI个人风格探索")]:
            score = pd.to_numeric(subset[field].astype(str).str.extract(r"([1-5])")[0], errors="coerce")
            valid = score.dropna()
            rows.append([
                group, label, int(valid.count()), float(valid.mean()) if len(valid) else pd.NA,
                float(valid.median()) if len(valid) else pd.NA,
                float(valid.ge(4).mean()) if len(valid) else pd.NA,
            ])
    return pd.DataFrame(rows, columns=[GROUP, "体验", "有效样本", "均分", "中位数", "4-5分高兴趣率"])

In [ ]:
def screening_audit(df: pd.DataFrame) -> pd.DataFrame:
    work = pd.DataFrame({
        "Q2原始城市层级": df["city"].map(clean_text).replace("", "未回答"),
        GROUP: df["city"].map(city_group),
        "Q9原始选项": df["q9"].map(clean_text).replace("", "未回答"),
        "是否购买后未持续": df["q9"].map(is_lapsed_buyer),
    })
    work["是否最终纳入"] = work[GROUP].isin(GROUP_ORDER) & work["是否购买后未持续"]
    return work.groupby(
        ["Q2原始城市层级", GROUP, "Q9原始选项", "是否购买后未持续", "是否最终纳入"],
        dropna=False,
    ).size().rename("人数").reset_index().sort_values(["是否最终纳入", "人数"], ascending=[False, False])

In [ ]:
def safe_sheet_name(name: str, used: set[str]) -> str:
    base = re.sub(r"[\\/*?:\[\]]", "_", name)[:31] or "Sheet"
    candidate = base
    counter = 2
    while candidate in used:
        suffix = f"_{counter}"
        candidate = base[: 31 - len(suffix)] + suffix
        counter += 1
    used.add(candidate)
    return candidate

In [ ]:
def write_excel(sheets: dict[str, pd.DataFrame], path: Path) -> None:
    used: set[str] = set()
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for requested, table in sheets.items():
            name = safe_sheet_name(requested, used)
            table.to_excel(writer, sheet_name=name, index=False)
            ws = writer.book[name]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.font = Font(name="微软雅黑", size=11, bold=True, color="FFFFFF")
                cell.fill = PatternFill("solid", fgColor="8F315B")
            for cells in ws.columns:
                sample = list(cells)[:200]
                width = min(max((len(str(cell.value or "")) for cell in sample), default=8) + 2, 48)
                ws.column_dimensions[cells[0].column_letter].width = max(width, 10)
            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    header = str(ws.cell(1, cell.column).value or "")
                    if isinstance(cell.value, float) and any(token in header for token in ("占比", "率", "百分点", "强度")):
                        cell.number_format = "0.0%"

In [ ]:
def create_chart(common: pd.DataFrame, output_dir: Path) -> None:
    if plt is None or common.empty:
        return
    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
    plt.rcParams["axes.unicode_minus"] = False
    chart = common.set_index("指标")[["一线/新一线占比", "三四线/县城占比"]]
    ax = chart.plot.bar(figsize=(13, 6), color=["#8F315B", "#DD7AA4"])
    ax.set_ylim(0, 1)
    ax.set_xlabel("")
    ax.set_ylabel("组内占比")
    ax.set_title("购买过但未持续购买人群：共同需求与兴趣信号")
    ax.legend(title="城市组")
    plt.xticks(rotation=22, ha="right")
    plt.tight_layout()
    plt.savefig(output_dir / "共同需求与兴趣信号_城市对比.png", dpi=180)
    plt.close()

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description="3CE购买后未持续人群的城市共同需求断点分析")
    script_dir = Path(__file__).resolve().parent
    parser.add_argument(
        "data", type=Path, nargs="?",
        default=script_dir / "3CE问卷数据-2026-08-12.csv",
        help="原始CSV或Excel问卷文件",
    )
    parser.add_argument("--sheet", default=0, help="Excel工作表名称或序号")
    parser.add_argument(
        "-o", "--output", type=Path,
        default=script_dir / "城市层级_未持续购买_共同需求断点结果",
        help="输出文件夹",
    )
    args = parser.parse_args()

    if not args.data.exists():
        raise FileNotFoundError(f"找不到数据文件：{args.data}")
    args.output.mkdir(parents=True, exist_ok=True)
    sheet: str | int = int(args.sheet) if isinstance(args.sheet, str) and args.sheet.isdigit() else args.sheet
    raw = read_survey(args.data, sheet)
    df = resolve_columns(raw)
    df[GROUP] = df["city"].map(city_group)
    df["是否购买后未持续"] = df["q9"].map(is_lapsed_buyer)
    target = df[df[GROUP].isin(GROUP_ORDER) & df["是否购买后未持续"]].copy()

    group_counts = target[GROUP].value_counts()
    missing_groups = [group for group in GROUP_ORDER if int(group_counts.get(group, 0)) == 0]
    if missing_groups:
        audit = screening_audit(df)
        audit_path = args.output / "筛选核验_比较组样本不足.csv"
        audit.to_csv(audit_path, index=False, encoding="utf-8-sig")
        raise ValueError(
            "以下比较组没有命中样本：" + "、".join(missing_groups)
            + f"。已输出筛选核验表：{audit_path}"
        )

    coded = code_themes(target)
    common = common_need_table(coded)
    overview = pd.DataFrame([
        ["全部记录", len(df)],
        ["购买后未持续且属于两个比较组", len(target)],
        ["一线/新一线目标样本", int(group_counts.get("一线/新一线", 0))],
        ["三四线/县城目标样本", int(group_counts.get("三四线/县城", 0))],
    ], columns=["指标", "人数"])

    code_columns = [
        "id", "age", "city", GROUP, "q9", "reason", "brand_need", "q10b", "q11b",
        "kdrama_element", "brand_info", "q16b", "q16c",
    ] + [column for column in coded.columns if column.startswith("个性化_") or column.startswith("场景化_")] + METRICS
    # 去重但保留顺序。
    code_columns = list(dict.fromkeys(code_columns))

    sheets = {
        "样本概览": overview,
        "筛选核验": screening_audit(df),
        "共同需求检验": common,
        "主题来源拆分": source_signal_table(coded),
        "断点_Q10B流失原因": grouped_multi_table(coded, "q10b"),
        "升级_Q11B激活方式": grouped_multi_table(coded, "q11b"),
        "兴趣_Q13穿搭妆容": grouped_multi_table(coded, "kdrama_element"),
        "升级_Q16个性场景": grouped_score_table(coded),
        "个体编码明细": coded[code_columns],
        "目标样本原始明细": target,
    }
    xlsx = args.output / "3CE_城市层级_未持续购买_共同需求断点分析.xlsx"
    write_excel(sheets, xlsx)
    coded[code_columns].to_csv(
        args.output / "3CE_未持续购买_共同需求编码明细.csv",
        index=False,
        encoding="utf-8-sig",
    )
    create_chart(common, args.output)

    print("\n========== 购买后未持续：共同需求断点 ==========")
    print(overview.to_string(index=False))
    print("\n【共同需求检验】")
    print(common.to_string(index=False))
    print(f"\nExcel：{xlsx.resolve()}")
    if plt is None:
        print("提示：未安装matplotlib，已跳过PNG图表；Excel和CSV不受影响。")

In [ ]:
if __name__ == "__main__":
    main()